<a href="https://colab.research.google.com/github/EgemenYapucu/DataScienceExercises/blob/main/SparkHW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get -qq update > /tmp/apt.out
!apt-get install -y -qq openjdk-11-jdk-headless
!(wget -q --show-progress -nc https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz)
!tar xf spark-3.5.1-bin-hadoop3.tgz
try:
  import pyspark, findspark, delta
except:
  %pip install -q --upgrade pyspark==3.5.1
  %pip install -q findspark
  %pip install -q delta
  !pip install gcsfs

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
spark-3.5.1-bin-had 100%[===================>] 381.90M  5.51MB/s    in 51s     
  Preparing metadata (setup.py) ... done


In [ ]:
#!(wget -O /content/spark-3.2.1-bin-hadoop3.2/jars/gcs-connector-hadoop2-latest.jar  -q https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3.2-latest.jar)

In [ ]:
import findspark
import pyspark
import os
from pyspark.sql.functions import *

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

findspark.init()
MAX_MEMORY="8g"

spark = (pyspark.sql.SparkSession.builder.appName("MyApp")
    .config("spark.executor.memory", MAX_MEMORY)
    .config("spark.driver.memory", MAX_MEMORY)
    .getOrCreate()
    )

spark

In [ ]:
spark.stop()

In [ ]:
!wget https://storage.googleapis.com/bigdata_training/Titanic.parquet

--2025-06-12 11:01:42--  https://storage.googleapis.com/bigdata_training/Titanic.parquet
Resolving storage.googleapis.com (storage.googleapis.com)... 108.177.12.207, 74.125.26.207, 173.194.217.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|108.177.12.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 40013 (39K) [application/octet-stream]
Saving to: ‘Titanic.parquet’

Titanic.parquet     100%[===================>]  39.08K  --.-KB/s    in 0.02s   

2025-06-12 11:01:42 (1.82 MB/s) - ‘Titanic.parquet’ saved [40013/40013]



In [ ]:
df_passengers = spark.read.parquet("/content/Titanic.parquet")
df_passengers.show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|      

In [ ]:
df_passengers.groupBy("Fare").agg(min("Fare")).show()

+-------+---------+
|   Fare|min(Fare)|
+-------+---------+
| 8.5167|   8.5167|
|   15.5|     15.5|
| 133.65|   133.65|
| 29.125|   29.125|
|10.4625|  10.4625|
| 7.0458|   7.0458|
|  9.475|    9.475|
|11.1333|  11.1333|
|    0.0|      0.0|
| 7.7333|   7.7333|
|   73.5|     73.5|
|77.2875|  77.2875|
|   15.9|     15.9|
|   11.5|     11.5|
| 8.6833|   8.6833|
|41.5792|  41.5792|
|    9.5|      9.5|
| 8.4042|   8.4042|
|14.4542|  14.4542|
|14.4583|  14.4583|
+-------+---------+
only showing top 20 rows



In [ ]:
#Gemiden kurtulma durumuna göre ödenen en yüksek ve en düşük bilet fiyatını bulunuz.
from pyspark.sql.functions import max, min
df_passengers.groupBy("Survived")\
.agg(
    max("Fare").alias("Max_Fare"),
    min("Fare").alias("Min_Fare")
    ).show()

+--------+--------+--------+
|Survived|Max_Fare|Min_Fare|
+--------+--------+--------+
|       0|   263.0|     0.0|
|       1|512.3292|     0.0|
+--------+--------+--------+



In [ ]:
#Yolcuların(18 yaş ve altı çocuklar hariç) cinsiyete göre ortalama yaşlarını bulunuz.
df_passengers.where(col("Age")>18).groupBy("Sex")\
.agg(avg("Age").alias("AvgAge")).show()

+------+------------------+
|   Sex|            AvgAge|
+------+------------------+
|female| 33.90673575129534|
|  male|34.480366492146594|
+------+------------------+



In [ ]:
#Aynı kabinde yolculuk yapan erkek ve kadınların eşleşmesi yapılması için aşağıdaki sorgu yazılmışmış fakat sorguda ufak bir eksiklik var, eksik olan parçayı tamamlayınız.
df_passengers.alias("m") \
  .join(df_passengers.alias("f"),
    on=((col("m.Cabin")==col("f.Cabin")) & (col("m.Name") != col("f.Name"))),
    how="inner") \
  .where((col("m.Sex")=="male") & (col("f.Sex")=="female")) \
  .select(col("m.Cabin"),col("f.Name").alias("Kadın Yolcu"),col("m.Name").alias("Erkek Yolcu")).show(100,200)

+-----------+----------------------------------------------------------------------------------+------------------------------------------+
|      Cabin|                                                                       Kadın Yolcu|                               Erkek Yolcu|
+-----------+----------------------------------------------------------------------------------+------------------------------------------+
|C23 C25 C27|                                                    Fortune, Miss. Alice Elizabeth|            Fortune, Mr. Charles Alexander|
|C23 C25 C27|                                                        Fortune, Miss. Mabel Helen|            Fortune, Mr. Charles Alexander|
|        C83|                                      Harris, Mrs. Henry Birkhardt (Irene Wallach)|               Harris, Mr. Henry Birkhardt|
|    B58 B60|                                   Baxter, Mrs. James (Helene DeLaudeniere Chaput)|                  Baxter, Mr. Quigg Edmond|
|       C123|       

In [ ]:
#veriye yolcunun yaşına göre belirlenecek yeni bir kolon eklenmek isteniyor(AgeCategory). Kolon içeriğinde;
#0-10 yaş arası yolcular -> Çocuk
#11-18 yaş arası yolcular -> Genç
#19 ve üstü yolcular -> Yetişkin
#bilgisi yazılması isteniyor. Bu yeni kolonu ekleyiniz.
df_passengers = df_passengers.withColumn(
    "AgeCategory",
    when(col("Age") <=10, "Çocuk")
    .when((col("Age") >= 11) & (col("Age") <= 18), "Genç")
    .when(col("Age") >=19, "Yetişkin")
    .otherwise("NULL")
)
df_passengers.show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+-----------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|AgeCategory|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+-----------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|   Yetişkin|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|   Yetişkin|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|   Yetişkin|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|   Yetişkin|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|      

In [ ]:
#spark'ta bir dataframe sorgu içnide de kullanılabilir. AgeCategory(bir üst sorunın çıktısı olan kolon) kolonuna göre yolcu sayısını veren SQL sorgunu düzenleyiniz.
df_passengers.createOrReplaceTempView("passengers")
spark.sql("""SELECT AgeCategory, COUNT(*) AS NumofPass
          FROM passengers
          GROUP BY AgeCategory""").show()

+-----------+---------+
|AgeCategory|NumofPass|
+-----------+---------+
|       Genç|       75|
|      Çocuk|       64|
|   Yetişkin|      575|
|       NULL|      177|
+-----------+---------+

